# 12b — Rating Personality prep (5,000-user cohort)

Build dual-channel tensors for Rating Personality with a **personality-based virtual part**:

- **Channel B1 (real):** user rating one-hot — identical to `channel1` from notebook 09b
- **Channel B2 (imaginary):** per-user personality scalar \(\mu_i - \mu_{\mathrm{global}}\) on train positions only

| Part | Section |
|------|--------|
| Part 0 | Setup |
| Part 1 | Load 09b outputs |
| Part 2 | Channel B1 = channel1 |
| Part 3 | Channel B2 personality fill + `personality.npy` |
| Part 4 | Save |
| Part 5 | Verification |

**Prerequisites:** notebook 09b (`data/processed/` artifacts).

## Part 0 — Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent

out_dir = root / "data" / "processed"
assert out_dir.exists(), f"Missing {out_dir} — run notebook 09b first."

RATING_LEVELS = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0], dtype=np.float64)
K = len(RATING_LEVELS)
rating_to_idx = {float(r): i for i, r in enumerate(RATING_LEVELS)}

print(f"Project root: {root}")
print(f"Processed:    {out_dir}")
print(f"K={K}, RATING_LEVELS={RATING_LEVELS.tolist()}")

Project root: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys
Processed:    /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed
K=10, RATING_LEVELS=[0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]


## Part 1 — Load 09b outputs

`mu_global` is the mean of all **training** ratings (count-weighted), not `mean(user_means)`.  
Recomputed from `channel1` + `mask` if `mu_global.npy` is absent.

In [2]:
# Bootstrap if Part 0 was skipped
try:
    out_dir
except NameError:
    from pathlib import Path
    import numpy as np
    root = Path.cwd().resolve()
    if root.name == "notebooks":
        root = root.parent
    out_dir = root / "data" / "processed"
    RATING_LEVELS = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0], dtype=np.float64)
    K = len(RATING_LEVELS)
    rating_to_idx = {float(r): i for i, r in enumerate(RATING_LEVELS)}
    print(f"(Part 0 skipped — bootstrapped out_dir={out_dir})")

path_cohort = out_dir / "cohort_user_ids.npy"
path_vocab = out_dir / "movie_vocab.npy"
path_mask = out_dir / "mask.npy"
path_ch1 = out_dir / "channel1_softmax.npy"
path_means = out_dir / "user_means.npy"
path_mu_g = out_dir / "mu_global.npy"

for p in (path_cohort, path_vocab, path_mask, path_ch1, path_means):
    assert p.exists(), f"Missing {p} — run notebook 09b first."

cohort_user_ids = np.load(path_cohort)
movie_vocab = np.load(path_vocab)
mask = np.load(path_mask, mmap_mode="r")
channel1 = np.load(path_ch1, mmap_mode="r")
user_means = np.load(path_means)  # float64 (n_users,)

n_users = len(cohort_user_ids)
n_movies = len(movie_vocab)
user_to_row = {int(uid): i for i, uid in enumerate(cohort_user_ids)}
movie_to_col = {int(mid): j for j, mid in enumerate(movie_vocab)}

assert channel1.shape == (n_users, n_movies, K), channel1.shape
assert mask.shape == (n_users, n_movies), mask.shape
assert user_means.shape == (n_users,), user_means.shape

if path_mu_g.exists():
    mu_global = float(np.load(path_mu_g))
    print(f"Loaded mu_global from {path_mu_g.name}")
else:
    # Recompute count-weighted global mean from train one-hots (same as 09b Part 4)
    print("Recomputing mu_global from channel1 + mask …")
    train_sum = 0.0
    train_cnt = 0
    step = 250
    for i0 in range(0, n_users, step):
        i1 = min(i0 + step, n_users)
        m_block = np.asarray(mask[i0:i1])
        c_block = np.asarray(channel1[i0:i1])
        for ii in range(i1 - i0):
            cols = np.where(m_block[ii] == 1)[0]
            if len(cols) == 0:
                continue
            ks = c_block[ii, cols, :].argmax(axis=1)
            ratings = RATING_LEVELS[ks]
            train_sum += float(ratings.sum())
            train_cnt += len(ratings)
    mu_global = train_sum / train_cnt
    np.save(path_mu_g, np.array(mu_global, dtype=np.float64))
    print(f"Saved: {path_mu_g}")

print(f"cohort_user_ids: {cohort_user_ids.shape}  {cohort_user_ids.dtype}")
print(f"movie_vocab:     {movie_vocab.shape}  {movie_vocab.dtype}")
print(f"mask:            {mask.shape}  {mask.dtype}")
print(f"channel1:        {channel1.shape}  {channel1.dtype}")
print(f"user_means:      {user_means.shape}  {user_means.dtype}")
print(f"mu_global:       {mu_global:.6f}")
print(f"(mean of user_means = {float(user_means.mean()):.6f} — unweighted; differs from mu_global)")


Loaded mu_global from mu_global.npy
cohort_user_ids: (5000,)  int64
movie_vocab:     (13129,)  int64
mask:            (5000, 13129)  int8
channel1:        (5000, 13129, 10)  float32
user_means:      (5000,)  float64
mu_global:       3.328083
(mean of user_means = 3.358298 — unweighted; differs from mu_global)


## Part 2 — Channel B1 (real part: user rating)

`channelB1` is identical to `channel1` from 09b — training-only one-hots; test / unrated stay zero.

In [3]:
channelB1 = channel1  # same array / memmap reference — no recomputation
print(f"channelB1 is channel1: {channelB1 is channel1}")
print(f"channelB1.shape = {channelB1.shape}  dtype={channelB1.dtype}")

channelB1 is channel1: True
channelB1.shape = (5000, 13129, 10)  dtype=float32


## Part 3 — Channel B2 (imaginary part: personality)

\(\mathrm{personality}_i = \mu_i - \mu_{\mathrm{global}}\)  
(positive = generous, negative = strict).

`channelB2[i, j, 0] = personality[i]` where `mask[i, j] == 1`, else `0.0`.  
Shape `(n_users, n_movies, 1)` float32 — **not** one-hot (scalar personality).

In [4]:
import shutil

personality = (user_means - mu_global).astype(np.float64)
assert personality.shape == (n_users,)

path_b2 = out_dir / "channelB2_personality.npy"
if path_b2.exists():
    path_b2.unlink()

bytes_b2 = int(n_users * n_movies * 1 * 4 + 20_000_000)
free = shutil.disk_usage(out_dir).free
print(f"Disk check for channelB2: need ~{bytes_b2 / 1e9:.2f} GB, free {free / 1e9:.2f} GB")
if free < bytes_b2:
    raise OSError(f"Not enough disk for channelB2 (~{bytes_b2 / 1e9:.2f} GB needed).")

# Build on disk via memmap
channelB2 = np.lib.format.open_memmap(
    path_b2, mode="w+", dtype=np.float32, shape=(n_users, n_movies, 1)
)
step = 250
for i0 in range(0, n_users, step):
    i1 = min(i0 + step, n_users)
    channelB2[i0:i1] = 0.0
    m_block = np.asarray(mask[i0:i1])  # (b, n_movies)
    for ii in range(i1 - i0):
        cols = np.where(m_block[ii] == 1)[0]
        if len(cols):
            channelB2[i0 + ii, cols, 0] = np.float32(personality[i0 + ii])
channelB2.flush()

print(f"personality: min={personality.min():.4f}, max={personality.max():.4f}, "
      f"median={float(np.median(personality)):.4f}, std={personality.std():.4f}")

bins = [(-np.inf, -1.0), (-1.0, -0.5), (-0.5, 0.0), (0.0, 0.5), (0.5, 1.0), (1.0, np.inf)]
labels = ["< -1.0", "[-1.0, -0.5)", "[-0.5, 0)", "[0, 0.5)", "[0.5, 1.0]", "> 1.0"]
print("\nPersonality histogram:")
for (lo, hi), lab in zip(bins, labels):
    if np.isneginf(lo):
        cnt = int(np.sum(personality < hi))
    elif np.isposinf(hi):
        cnt = int(np.sum(personality > lo))
    elif hi == 1.0 and lo == 0.5:
        cnt = int(np.sum((personality >= lo) & (personality <= hi)))
    else:
        cnt = int(np.sum((personality >= lo) & (personality < hi)))
    print(f"  {lab:>14s}: {cnt:,}")

print(f"\nchannelB2.shape = {channelB2.shape}  dtype={channelB2.dtype}")

Disk check for channelB2: need ~0.28 GB, free 8.68 GB
personality: min=-2.2450, max=1.5947, median=0.0571, std=0.4143

Personality histogram:
          < -1.0: 77
    [-1.0, -0.5): 414
       [-0.5, 0): 1,741
        [0, 0.5): 2,195
      [0.5, 1.0]: 551
           > 1.0: 22

channelB2.shape = (5000, 13129, 1)  dtype=float32


## Part 4 — Save

`channelB1_softmax.npy` is a hard link to `channel1_softmax.npy` when possible (identical bytes, no extra ~2.6 GB).  
`mask.npy` already written by 09b.

In [5]:
import os

path_b1 = out_dir / "channelB1_softmax.npy"
path_pers = out_dir / "personality.npy"

assert path_mask.exists(), f"mask.npy missing at {path_mask}"

# channelB1: hard-link to channel1 (same content); fall back to copy if link fails
if path_b1.exists() or path_b1.is_symlink():
    path_b1.unlink()
try:
    os.link(path_ch1, path_b1)
    print(f"Saved: {path_b1}  (hard link → {path_ch1.name})")
except OSError as e:
    print(f"Hard link failed ({e}); copying (needs ~{channel1.nbytes / 1e9:.2f} GB free) …")
    free = shutil.disk_usage(out_dir).free
    if free < channel1.nbytes + 50_000_000:
        raise OSError(
            f"Not enough disk to copy channelB1. Need ~{channel1.nbytes / 1e9:.2f} GB, free {free / 1e9:.2f} GB."
        )
    shutil.copy2(path_ch1, path_b1)
    print(f"Saved: {path_b1}  (copy of {path_ch1.name})")

np.save(path_pers, personality)
# channelB2 already written via memmap in Part 3

print(f"Saved: {path_b2}  shape={channelB2.shape}  ({path_b2.stat().st_size / 1e6:.1f} MB)")
print(f"Saved: {path_pers}  shape={personality.shape}  dtype={personality.dtype}")
print(f"Asserted present: {path_mask.name}")

Saved: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/channelB1_softmax.npy  (hard link → channel1_softmax.npy)
Saved: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/channelB2_personality.npy  shape=(5000, 13129, 1)  (262.6 MB)
Saved: /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/data/processed/personality.npy  shape=(5000,)  dtype=float64
Asserted present: mask.npy


## Part 5 — Verification

In [6]:
# B1 identical to channel1 (same object, or equal on disk)
assert channelB1 is channel1 or np.array_equal(np.asarray(channelB1[:1]), np.asarray(channel1[:1]))
b1_disk = np.load(path_b1, mmap_mode="r")
assert b1_disk.shape == channel1.shape
# Spot-check equality on a few users (full compare would re-read 2.6GB)
for i in (0, n_users // 2, n_users - 1):
    assert np.array_equal(np.asarray(b1_disk[i]), np.asarray(channel1[i])), f"B1 mismatch at row {i}"
print("✓ channelB1 identical to channel1 (reference + disk spot-check)")

# B2 nonzero positions == mask == 1
step = 250
for i0 in range(0, n_users, step):
    i1 = min(i0 + step, n_users)
    m_block = np.asarray(mask[i0:i1])
    b2_block = np.asarray(channelB2[i0:i1, :, 0])
    nonzero = b2_block != 0.0
    train = m_block == 1
    # train cells must equal personality; non-train must be 0
    # (personality can be exactly 0.0 for some users — those train cells are also 0, so nonzero ⊆ train)
    assert not np.any(nonzero & ~train), "B2 nonzero outside mask==1"
    for ii in range(i1 - i0):
        cols = np.where(train[ii])[0]
        if len(cols) == 0:
            continue
        expected = np.float32(personality[i0 + ii])
        got = b2_block[ii, cols]
        assert np.allclose(got, expected), (
            f"B2 contamination at user row {i0 + ii}: expected {expected}, got unique={np.unique(got)}"
        )
        # unrated/test must be 0
        assert np.all(b2_block[ii, m_block[ii] == 0] == 0.0)
print("✓ channelB2 nonzero ⊆ mask==1; each user row is {{0}} ∪ {personality[i]}")

# Most generous / most strict
i_gen = int(np.argmax(personality))
i_strict = int(np.argmin(personality))
for label, i in (("most generous", i_gen), ("most strict", i_strict)):
    uid = int(cohort_user_ids[i])
    n_train = int(np.asarray(mask[i]).sum())
    print(
        f"\n{label}: userId={uid}  mu_user={user_means[i]:.4f}  "
        f"personality={personality[i]:+.4f}  n_train_movies={n_train}"
    )

# Sample user with personality > 0.5
cands = np.where(personality > 0.5)[0]
assert len(cands) > 0, "No user with personality > 0.5"
i_s = int(cands[0])
uid_s = int(cohort_user_ids[i_s])
train_js = np.where(np.asarray(mask[i_s]) == 1)[0]
unrated_js = np.where(np.asarray(mask[i_s]) == 0)[0]
print(f"\nSample userId={uid_s}  personality={personality[i_s]:+.4f}")
print("  channelB2 on 3 train movies:")
for j in train_js[:3]:
    print(f"    movieId={int(movie_vocab[j])}  B2={float(channelB2[i_s, j, 0]):+.6f}")
j_u = int(unrated_js[0])
print(f"  channelB2 on 1 unrated/test movie: movieId={int(movie_vocab[j_u])}  B2={float(channelB2[i_s, j_u, 0]):+.6f}")
print("\nAll checks passed.")

✓ channelB1 identical to channel1 (reference + disk spot-check)
✓ channelB2 nonzero ⊆ mask==1; each user row is {{0}} ∪ {personality[i]}

most generous: userId=48498  mu_user=4.9228  personality=+1.5947  n_train_movies=764

most strict: userId=113857  mu_user=1.0831  personality=-2.2450  n_train_movies=698

Sample userId=125794  personality=+0.5246
  channelB2 on 3 train movies:
    movieId=1  B2=+0.524613
    movieId=2  B2=+0.524613
    movieId=5  B2=+0.524613
  channelB2 on 1 unrated/test movie: movieId=3  B2=+0.000000

All checks passed.
